In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

INPUT_ROOT = Path("/kaggle/input")
REPO_DIR = Path("/kaggle/working/MASI")
STORAGE_ROOT = Path("/kaggle/working/MASI_workdir")

candidates = []
for pyproject_path in INPUT_ROOT.glob("**/pyproject.toml"):
    candidate = pyproject_path.parent
    if (candidate / "scripts" / "train_masi.py").exists() and (candidate / "src" / "masi").exists():
        candidates.append(candidate)

if not candidates:
    raise FileNotFoundError("Could not find the uploaded MASI repository under /kaggle/input.")

repo_source = sorted(candidates, key=lambda path: (len(path.parts), str(path)))[0]
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(repo_source, REPO_DIR)
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
print(f"Copied repo from {repo_source} to {REPO_DIR}")
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[recommender]"], check=True)


In [ ]:
import os
import subprocess

os.chdir("/kaggle/working/MASI")
subprocess.run(
    "PYTHONPATH=src python scripts/download_amazon_csj_dataset.py --output-dir /kaggle/working/MASI_workdir/data/raw/amazon_reviews_2023 --review-range-end 6442450943 --download-metadata",
    shell=True,
    check=True,
)


In [ ]:
import os
import subprocess

os.chdir("/kaggle/working/MASI")
subprocess.run(
    "PYTHONPATH=src python scripts/download_amazon_csj_images.py --config configs/masi_train_csj_medium_kaggle.json --storage-root /kaggle/working/MASI_workdir --workers 16 --retries 3 --resume",
    shell=True,
    check=True,
)


In [ ]:
import os
import subprocess

os.chdir("/kaggle/working/MASI")
subprocess.run(
    "PYTHONPATH=src python scripts/train_masi.py --config configs/masi_train_csj_medium_kaggle.json --storage-root /kaggle/working/MASI_workdir",
    shell=True,
    check=True,
)


In [ ]:
from pathlib import Path
import shutil
import subprocess

RUN_ROOT = Path("/kaggle/working/MASI_workdir/outputs/amazon_csj_medium_kaggle_train")
EXPORT_ROOT = Path("/kaggle/working/MASI_outputs")
ZIP_PATH = Path("/kaggle/working/MASI_outputs.zip")

if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

shutil.copy2(RUN_ROOT / "run_manifest.json", EXPORT_ROOT / "run_manifest.json")
for directory_name in ["resolved_configs", "checkpoints", "phase12_tokens", "phase3_experiment"]:
    shutil.copytree(RUN_ROOT / directory_name, EXPORT_ROOT / directory_name)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
subprocess.run(["zip", "-r", str(ZIP_PATH), str(EXPORT_ROOT)], check=True)
print(f"Wrote {ZIP_PATH}")
